# Lab-8: Deep Learning with Lightning

This lab is design for investigating the basic architecture and parameter searching techniques in deep learning literature. We use a fundamental dataset, MNIST, with the help of PyTorch Lightning and design our model to handle image classification task.

### General Announcements

- The exercises on this sheet are graded by a maximum of **14 points**. You will be asked to implement several functions.
- Team work is not allowed! Everybody implements his/her own code. Discussing issues with others is fine, sharing code with others is not.
- If you use any code fragments found on the Internet, make sure you reference them properly.
- You can send your questions via email to the TAs until the deadline.

### Suggestions

Please install pytorch lightning packages via conda or pip before starting the lab session. You can use the tutorials of [Pytorch Lightning](https://lightning.ai/docs/pytorch/stable/notebooks/lightning_examples/mnist-hello-world.html).

Please also check: [Tensorboard](https://pytorch.org/tutorials/recipes/recipes/tensorboard_with_pytorch.html)

For installing lightning: [Link](https://lightning.ai/pytorch-lightning)


In [38]:
# Basic Machine Learning Modules
import pandas
import numpy
import sklearn

# Deep Learning Modules
import torch
import lightning.pytorch as pl

# Visualization Modules
import matplotlib.pyplot as plt

# Others
from torchvision.datasets import MNIST
from torchvision import transforms
from torchmetrics import Accuracy
from lightning.pytorch.loggers import CSVLogger, TensorBoardLogger
from lightning.pytorch.callbacks.early_stopping import EarlyStopping
from lightning.pytorch.callbacks import TQDMProgressBar
import warnings

warnings.filterwarnings("ignore")


DATASET_PATH = "./MNIST/"  # You can change them
EXPERIMENTS_PATH = "./MNIST_EXP"  # You can change them

# 1) Dataset (3 Points)


- Design a DataModule for MNIST.
- Use 80/20 % split for train/val sets.


In [39]:
class MNISTDataModule(pl.LightningDataModule):
    def __init__(self, data_folder: str = DATASET_PATH, batch_size: int = 64, num_cpu: int = 1):
        super().__init__()
        self.path = data_folder
        self.batch_size = batch_size
        self.num_cpu = num_cpu
        self.transform = transforms.Compose(
            [transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))]
        )
        self.train_ratio = 0.80
        self.val_ratio = 0.20

    def prepare_data(self) -> None:
        # Download MNIST Data in self.path
        self.mnist_trainset = MNIST(root=self.path, train=True, download=True, transform=self.transform)
        self.mnist_testset = MNIST(root=self.path, train=False, download=True, transform=self.transform)

        # Split train set to actual train set and validation set
        train_split_ratio = int(self.mnist_trainset.train_data.shape[0] * self.train_ratio)
        val_split_ratio = int(self.mnist_trainset.train_data.shape[0] - train_split_ratio)
        self.train_set, self.val_set = torch.utils.data.random_split(
            self.mnist_trainset.train_data, [train_split_ratio, val_split_ratio]
        )

    def setup(self, stage: str = "fit") -> None:
        if stage in ["fit", "tune"]:
            self.train_dataset = self.train_set

        if stage in ["fit", "tune", "validate"]:
            self.val_dataset = self.val_set

        elif stage in ["test", "predict"]:
            self.test_dataset = self.mnist_testset.test_data

        else:
            raise NotImplementedError("Unknown Stage: {}".format(stage))

    def train_dataloader(self) -> torch.utils.data.DataLoader:
        train_dataloader = torch.utils.data.DataLoader(
            dataset=self.train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_cpu,
            pin_memory=True,
        )
        return train_dataloader

    def val_dataloader(self) -> torch.utils.data.DataLoader:
        val_dataloader = torch.utils.data.DataLoader(
            self.val_dataset,
            self.batch_size,
            shuffle=True,
            #  drop_last=True,
            num_workers=self.num_cpu,
            pin_memory=True,
        )
        return val_dataloader

    def test_dataloader(self) -> torch.utils.data.DataLoader:
        test_dataloader = torch.utils.data.DataLoader(
            self.test_dataset,
            self.batch_size,
            shuffle=True,
            #  drop_last=True,
            num_workers=self.num_cpu,
            pin_memory=True,
        )
        return test_dataloader

    def predict_dataloader(self) -> torch.utils.data.DataLoader:
        return self.test_dataloader()


data_module = MNISTDataModule()
data_module.prepare_data()
data_module.setup("fit")
dl = data_module.train_dataloader()
print(next(dl.__iter__()))

tensor([[[0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         ...,
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0]],

        [[0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         ...,
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0]],

        [[0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         ...,
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0]],

        ...,

        [[0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         ...,
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0]],

        [[0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0,

In [40]:
# Do not change! Only for checking
print("Shape of Images: [B x C x H x W] = ", next(dl.__iter__())[0].shape)
print("Shape of Labels: [B] = ", next(dl.__iter__())[1].shape)

Shape of Images: [B x C x H x W] =  torch.Size([28, 28])
Shape of Labels: [B] =  torch.Size([28, 28])


## 2) Neural Network Architecture (2 Points)


- Design an model with 3 Convolutional layers and 1 fully-connected layer, in this order:
  - Convolution: Kernel size = 3x3, padding = 'same', number of filters = 4
  - Convolution: Kernel size = 3x3, padding = 'same', number of filters = 8
  - Convolution: Kernel size = 3x3, padding = 'same', number of filters = 4
  - Linear: No Bias, num_class = 10 in MNIST Dataset
- Use rectified linear unit (ReLU) for activation function.
- Initialize all the weights with `xavier_uniform`.


In [60]:
class RawModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.my_layer = torch.nn.Sequential(
            torch.nn.Conv2d(1, 4, 3, padding="same"),
            torch.nn.ReLU(),
            torch.nn.Conv2d(4, 8, 3, padding="same"),
            torch.nn.ReLU(),
            torch.nn.Conv2d(8, 4, 3, padding="same"),
            torch.nn.ReLU()
        )
        self.model = torch.nn.Linear(4 * 28 * 28, 10)
 
        self.model.apply(self.initialize_weights)
    
    @staticmethod
    def initialize_weights(module: torch.nn.Module) -> None:
        torch.nn.init.xavier_uniform_(module.weight)

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        """
        Input: torch.Tensor | dtype=torch.float | shape=[B, C, H, W]
        Output: torch.Tensor | dtype=torch.float | shape=[B, num_class]
        """
        images = self.my_layer(images)
        images = images.view(images.size(0), -1)
        images = self.model(images)
        return images


In [61]:
model = RawModel()
# model.apply(initialize_weights)
with torch.no_grad():
    sample_image = torch.rand(size=(4, 1, 28, 28))
    output = model(sample_image)
    print(output.shape, output.dtype)

torch.Size([4, 10]) torch.float32


## 3) Experiments (4 Points)


- Define training/validation/test step and optimizer with cross-entropy loss and Adam optimizer.
- Use accuracy scores for monitoring the experiment. (multiclass accuracy from Lightning Metrics)


In [59]:
class MNISTExperiment(pl.LightningModule):
    def __init__(self, learning_rate: float = 1e-3):
        super().__init__()
        self.model = RawModel()
        self.learning_rate = learning_rate

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x)

    def training_step(self, batch) -> torch.Tensor:
        x, y = batch
        y_hat = self(x)
        loss = torch.nn.functional.cross_entropy(y_hat, y)
        accuracy = pl.metrics.functional.accuracy(y_hat, y)
        self.log("train_loss", loss)
        self.log("train_accuracy", accuracy)
        return loss

    def validation_step(self, batch) -> None:
        x, y = batch
        y_hat = self(x)
        loss = torch.nn.functional.cross_entropy(y_hat, y)
        preds = torch.argmax(y_hat, dim=1)
        accuracy = pl.metrics.functional.accuracy(preds, y)
        self.log('validation_loss', loss, prog_bar=True)
        self.log('validation_accuracy', accuracy, prog_bar=True)

    def test_step(self, batch) -> None: 
        x, y = batch
        y_hat = self(x)
        preds = torch.argmax(y_hat, dim=1)
        loss = torch.nn.functional.cross_entropy(y_hat, y)
        accuracy = pl.metrics.functional.accuracy(preds, y)
        self.log('test_accuracy', accuracy)
        self.log('test_loss', loss)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(lr=self.learning_rate)
        return optimizer

experiment = MNISTExperiment()

with torch.no_grad():
    sample_image = torch.rand(size=(4, 1, 28, 28))
    output = experiment(sample_image)
    print(output.shape, output.dtype)

torch.Size([4, 10]) torch.float32


- Define a trainer of lightning.
  - Maximum epoch = 20
  - accelerator = 'auto'
  - Use `CSVLogger` and `TensorBoardLogger`
  - Use `EarlyStopping` with patience epoch = 3
  - Use `TQDMProgressBar` with refresh rate = 10


In [47]:
# Max epoch
max_epoch = 20
# accelerator
acce = "auto"
# logger & TensorBoardLogger
csv_logger = CSVLogger("logs")
tb_logger = TensorBoardLogger("logs")
# Early stopping
patience_val = 3
early_stop_callback = EarlyStopping(
    monitor="val_loss", 
    patience=patience_val, 
    mode="min"
    )
# Progress bar
# It seems that progress bar is deprecated in the latest stable version of lightning, no?
prog_refresh_rate = 10
progress_bar = TQDMProgressBar(refresh_rate=prog_refresh_rate)

# My trainer
trainer = pl.Trainer(
    logger=[csv_logger, tb_logger],
    callbacks=[early_stop_callback],
    max_epochs=max_epoch,
    accelerator=acce,
    # progress_bar_refresh_rate=prog_refresh_rate,
    # progress_bar=progress_bar
    )

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


## 4) Results (4 Points)

- Train and test the `RawModel` and plot the score and loss values versus epoch.


In [ ]:
trainer.fit(model)

- Re-design the model with dropout layer in-between the convolutional layers and re-train the model. as like `RawModel` and create a new module as `ModelWithDropout`. Try:
  - dropout probability = 0.1
  - dropout probability = 0.5
  - dropout probability = 0.9


In [62]:
class RawModel(torch.nn.Module):
    def __init__(self,dropout):
        super().__init__()
        self.my_layer = torch.nn.Sequential(
            torch.nn.Conv2d(1, 4, 3, padding="same"),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Conv2d(4, 8, 3, padding="same"),
            torch.nn.ReLU(),
            torch.nn.Conv2d(8, 4, 3, padding="same"),
            torch.nn.ReLU()
        )
        self.model = torch.nn.Linear(4 * 28 * 28, 10)
 
        self.model.apply(self.initialize_weights)
    
    @staticmethod
    def initialize_weights(module: torch.nn.Module) -> None:
        torch.nn.init.xavier_uniform_(module.weight)

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        """
        Input: torch.Tensor | dtype=torch.float | shape=[B, C, H, W]
        Output: torch.Tensor | dtype=torch.float | shape=[B, num_class]
        """
        images = self.my_layer(images)
        images = images.view(images.size(0), -1)
        images = self.model(images)
        return images

In [63]:
ModelWithDropout = RawModel(dropout=0.1)
with torch.no_grad():
    sample_image = torch.rand(size=(4, 1, 28, 28))
    output = model(sample_image)
    print(output.shape, output.dtype)

torch.Size([4, 10]) torch.float32


In [52]:
ModelWithDropout = RawModel(dropout=0.5)
with torch.no_grad():
    sample_image = torch.rand(size=(4, 1, 28, 28))
    output = model(sample_image)
    print(output.shape, output.dtype)

torch.Size([4, 10]) torch.float32


In [53]:
ModelWithDropout = RawModel(dropout=0.9)
with torch.no_grad():
    sample_image = torch.rand(size=(4, 1, 28, 28))
    output = model(sample_image)
    print(output.shape, output.dtype)

torch.Size([4, 10]) torch.float32


- Re-design the model with batchnorm layer in-between the convolutional layers and re-train the model.


In [54]:
class RawModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.my_layer = torch.nn.Sequential(
            torch.nn.Conv2d(1, 4, 3, padding="same"),
            torch.nn.BatchNorm2d(4),
            torch.nn.ReLU(),
            torch.nn.Conv2d(4, 8, 3, padding="same"),
            torch.nn.BatchNorm2d(8),
            torch.nn.ReLU(),
            torch.nn.Conv2d(8, 4, 3, padding="same"),
            torch.nn.ReLU()
        )
        self.model = torch.nn.Linear(4 * 28 * 28, 10)
 
        self.model.apply(self.initialize_weights)
    
    @staticmethod
    def initialize_weights(module: torch.nn.Module) -> None:
        torch.nn.init.xavier_uniform_(module.weight)

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        """
        Input: torch.Tensor | dtype=torch.float | shape=[B, C, H, W]
        Output: torch.Tensor | dtype=torch.float | shape=[B, num_class]
        """
        images = self.my_layer(images)
        images = images.view(images.size(0), -1)
        images = self.model(images)
        return images

In [55]:
ModelWithDropout = RawModel()
with torch.no_grad():
    sample_image = torch.rand(size=(4, 1, 28, 28))
    output = model(sample_image)
    print(output.shape, output.dtype)

torch.Size([4, 10]) torch.float32


## 5) Conclusion (1 Point)

Comment on your findings:

- Show the results in a table via Pandas
- Which method is better? Why?
- Are the results significant? If not, how can we get significant ones?


ANSWER: ...
